## 1. Imports

In [1]:
import requests
import pandas as pd
import time
import json

from pathlib import Path
from datetime import datetime

## 2. Funções

In [2]:
# funçoes para coletar os dados da API

LIMITE_TOTAL_EDITAIS = 3000

def coletarEditaisModalidade(codigo_modalidade, data_inicial, data_final, limite_restante):
    #Coleta editais de uma modalidade específica, respeitando um limite máximo de registros a coletar

    editais = []
    pagina = 1
    
    while True:
        if len(editais) >= limite_restante:
            print(f"Limite de coleta atingido para esta modalidade. Parando.")
            break
        
        params = {
            "dataInicial": data_inicial,
            "dataFinal": data_final,
            "codigoModalidadeContratacao": codigo_modalidade,
            "pagina": pagina,
            "tamanhoPagina": TAMANHO_PAGINA
        }
        
        response = requests.get(BASE_URL, params=params)
        
        if response.status_code == 429:
            print(f"Rate limit atingido na página {pagina}. Esperando 5s ...")
            time.sleep(5)
            continue
        
        if response.status_code != 200:
            print(f"Erro na modalidade {codigo_modalidade}, na página {pagina}: status {response.status_code}")
            break
        
        dados = response.json()
        registros = dados.get("data", [])
        
        if not registros:
            break
        
        editais.extend(registros)
        
        total_paginas = dados.get("totalPaginas", 1)
        print(f"Modalidade {codigo_modalidade} - página {pagina}/{total_paginas} - {len(registros)} registros (acumulado: {len(editais)})")
        
        if pagina >= total_paginas:
            break
        
        pagina += 1
        time.sleep(1.5)
    
    return editais


def coletarTodasModalidades(modalidades, data_inicial, data_final, limite_total):
    #Roda a coleta para cada modalidade, parando quando o limite GLOBAL é atingido. nao vi ter um numero certo para cada modalidade, para trazer o retrato da realidade

    todos_editais = []
    
    for codigo, nome in modalidades.items():
        limite_restante = limite_total - len(todos_editais)
        
        if limite_restante <= 0:
            print(f"\nLimite total de {limite_total} já atingido. Pulando modalidade {nome}.")
            break
        
        print(f"\n- Coletando: {nome} (código {codigo}) - restam {limite_restante} para o limite total -")
        editais_modalidade = coletarEditaisModalidade(codigo, data_inicial, data_final, limite_restante)
        
        for edital in editais_modalidade:
            edital["modalidade_nome"] = nome
        
        todos_editais.extend(editais_modalidade)
        print(f"Total coletado para {nome}: {len(editais_modalidade)} | Total geral acumulado: {len(todos_editais)}")
    
    return todos_editais

## 3. Coleta de dados - API do PNCP (Potal Nascional de Contas Publicas)

In [3]:
# configuraçoes essenciais para puxar os dados certos da API

BASE_URL = "https://pncp.gov.br/api/consulta/v1/contratacoes/publicacao"
DADOS_RAW_PATH = Path("../dados/raw")
DADOS_RAW_PATH.mkdir(parents=True, exist_ok=True)

DATA_INICIAL = "20260601"
DATA_FINAL = "20260603"

MODALIDADES = {
    4: "Concorrência - Eletrônica",
    5: "Concorrência - Presencial",
    6: "Pregão - Eletrônico",
    7: "Pregão - Presencial",
    8: "Dispensa de Licitação",
    9: "Inexigibilidade"
}

TAMANHO_PAGINA = 50

In [4]:
# coletando os dados 

todos_editais = coletarTodasModalidades( modalidades=MODALIDADES,
                                        data_inicial= DATA_INICIAL,
                                        data_final= DATA_FINAL,
                                        limite_total= LIMITE_TOTAL_EDITAIS )

print(f"Total de editais coletados: {len(todos_editais)}")


- Coletando: Concorrência - Eletrônica (código 4) - restam 3000 para o limite total -
Modalidade 4 - página 1/22 - 50 registros (acumulado: 50)
Modalidade 4 - página 2/22 - 50 registros (acumulado: 100)
Modalidade 4 - página 3/22 - 50 registros (acumulado: 150)
Modalidade 4 - página 4/22 - 50 registros (acumulado: 200)
Modalidade 4 - página 5/22 - 50 registros (acumulado: 250)
Modalidade 4 - página 6/22 - 50 registros (acumulado: 300)
Modalidade 4 - página 7/22 - 50 registros (acumulado: 350)
Modalidade 4 - página 8/22 - 50 registros (acumulado: 400)
Modalidade 4 - página 9/22 - 50 registros (acumulado: 450)
Modalidade 4 - página 10/22 - 50 registros (acumulado: 500)
Modalidade 4 - página 11/22 - 50 registros (acumulado: 550)
Modalidade 4 - página 12/22 - 50 registros (acumulado: 600)
Modalidade 4 - página 13/22 - 50 registros (acumulado: 650)
Modalidade 4 - página 14/22 - 50 registros (acumulado: 700)
Modalidade 4 - página 15/22 - 50 registros (acumulado: 750)
Modalidade 4 - página 1

In [5]:
print(type(len))

<class 'builtin_function_or_method'>


### 3.1 Salvando o Dataset Bruto

In [6]:
caminho_arquivo = DADOS_RAW_PATH / "editais.json"

with open(caminho_arquivo, "w", encoding="utf-8") as f:
    json.dump(todos_editais, f, ensure_ascii=False, indent=2)

print(f"salvos {len(todos_editais)} editais em {caminho_arquivo}")


salvos 3011 editais em ../dados/raw/editais.json


### 3.2 Transformando em DataFrame

In [7]:
df = pd.DataFrame(todos_editais)

# aproeitando o rotulo que ja veem para categorizar os dados
df['categoria'] = df['modalidade_nome']

print(f"total de registros {len(df)}")
print( df['categoria'].value_counts())

total de registros 3011
categoria
Pregão - Eletrônico          1900
Concorrência - Eletrônica    1061
Concorrência - Presencial      50
Name: count, dtype: int64
